<a href="https://colab.research.google.com/github/cbonnin88/Wavestream/blob/main/data_generation_and_cleaning.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
import polars as pl
import numpy as np
import uuid
from datetime import datetime, timedelta

# **Settings**

In [2]:
NUM_USERS = 5000
NUM_TRACKS = 1000
START_DATE = datetime(2026,1,1)

# **Tracks Table**

In [3]:
genres = [
    'Lo-fi', 'Synthwave', 'Phonk', 'Indie Rock', 'Techno',
    'Afrobeats', 'Reggaeton', 'K-Pop', 'Heavy Metal', 'Jazz Fusion',
    'Hyperpop', 'Amapiano', 'Classical', 'Folk', 'Deep House'
]

In [5]:
tracks = pl.DataFrame({
    'track_id': [str(uuid.uuid4())[:8] for _ in range(NUM_TRACKS)],
    'genre': np.random.choice(genres,NUM_TRACKS),
    'bpm': np.random.randint(60,200,NUM_TRACKS),
    'duration_sec': np.random.randint(90,450,NUM_TRACKS)
})

# **Users Table**

In [6]:
countries = ['France','Germany','United Kingdom','Spain','Norway','Austria','Luxembourg','Belgium','Ireland','The Netherlands']
sub_types = ['free','basic','premium']
price_map = {'free':0.00,'basic':7.99,'premium':15.99}

In [7]:
user_ids = [str(i) for i in range(NUM_USERS)]
user_subs = np.random.choice(sub_types,NUM_USERS, p=[0.5,0.3,0.2])

In [8]:
users = pl.DataFrame({
    'user_id':user_ids,
    'signup_date': [START_DATE + timedelta(days=np.random.randint(0,30)) for _ in range(NUM_USERS)],
    'country': np.random.choice(countries, NUM_USERS),
    'subscription_type': user_subs,
    'subscription_price': [price_map[s] for s in user_subs],
    'ab_group': np.random.choice(['control','treatment'],NUM_USERS)
})

# **Events Table**

In [9]:
event_types = ['Play','Skip','Add to Playlist','Search','Like']
event_probs = [0.45,0.15,0.05,0.25,0.1]
devices = ['Mobile','Desktop','Tablet','Smart TV']
device_probs = [0.65,0.20,0.10,0.05]

In [12]:
event_data = []

for row in users.iter_rows(named=True):
  # Tier-based activity levels
  if row['subscription_type'] == 'premium':
    num_events = np.random.randint(50,150)
  elif row['subscription_type'] == 'basic':
    num_events = np.random.randint(20,70)
  else:
    num_events = np.random.randint(5,30)

  primary_device = np.random.choice(devices, p=device_probs)

  for _ in range(num_events):
    event_time = row['signup_date'] + timedelta(
        days=np.random.randint(0,60),
        hours=np.random.randint(0,24),
        minutes=np.random.randint(0,60)
    )

    final_device = primary_device if np.random.random() > 0.15 else np.random.choice(devices)

    event_data.append({
        'event_id': str(uuid.uuid4()),
        'user_id': row['user_id'],
        'event_name': np.random.choice(event_types, p=event_probs),
        'track_id': np.random.choice(tracks['track_id']),
        'timestamp': event_time,
        'device_id': f'device_{row['user_id']}',
        'device_type': final_device
    })

events = pl.DataFrame(event_data).sort('timestamp')


In [13]:
display(tracks.head())

track_id,genre,bpm,duration_sec
str,str,i64,i64
"""3782be11""","""Techno""",146,449
"""f81ac934""","""K-Pop""",119,433
"""37e2ee8f""","""Jazz Fusion""",60,220
"""30b7c893""","""Afrobeats""",65,415
"""52d48584""","""K-Pop""",171,410


In [14]:
display(users.head())

user_id,signup_date,country,subscription_type,subscription_price,ab_group
str,datetime[μs],str,str,f64,str
"""0""",2026-01-07 00:00:00,"""France""","""free""",0.0,"""treatment"""
"""1""",2026-01-11 00:00:00,"""Germany""","""basic""",7.99,"""control"""
"""2""",2026-01-13 00:00:00,"""France""","""premium""",15.99,"""treatment"""
"""3""",2026-01-18 00:00:00,"""Belgium""","""free""",0.0,"""treatment"""
"""4""",2026-01-12 00:00:00,"""Austria""","""free""",0.0,"""treatment"""


In [15]:
display(events.head())

event_id,user_id,event_name,track_id,timestamp,device_id,device_type
str,str,str,str,datetime[μs],str,str
"""4386906a-f9ae-4587-ad18-f05d68…","""2616""","""Like""","""80f18d35""",2026-01-01 00:04:00,"""device_2616""","""Desktop"""
"""a3e56861-19ef-47b1-b024-408d38…","""2467""","""Skip""","""bdd5fd0b""",2026-01-01 00:05:00,"""device_2467""","""Mobile"""
"""3fda6cff-ebfe-44ea-a6f2-68e7fa…","""1386""","""Play""","""80c73fd8""",2026-01-01 00:08:00,"""device_1386""","""Desktop"""
"""e5afc88a-8a6d-4f57-b184-3fa14c…","""1231""","""Like""","""30f35810""",2026-01-01 00:11:00,"""device_1231""","""Smart TV"""
"""46daa17e-bb38-4381-9cc8-f7ec25…","""948""","""Like""","""0705d651""",2026-01-01 00:29:00,"""device_948""","""Desktop"""


In [17]:
print(f'Unique Genres: {len(genres)}')

Unique Genres: 15


In [18]:
print(f'Total Rows in Events: {events.height}')

Total Rows in Events: 210159


# **Cleaning up the Data**

In [19]:
# Convert timestamp to actual datetime
events = events.with_columns(pl.col('timestamp').cast(pl.Datetime))

In [21]:
#2. Sessionization Logic
# A. Sort by user and time
# B. Calculate time diff between current and previous event per user
# C. If diff > 30 mins, mark as a new session (1) else (0)
# D. Cumulative sum these marks to create a unique Session ID

events_cleaned = events.sort(['user_id','timestamp']).with_columns([
    (
        (pl.col('timestamp') - pl.col('timestamp').shift(1).over('user_id'))
        .dt.total_minutes() > 30
    ).fill_null(True).alias('is_new_session')
]).with_columns([
    (pl.struct(['user_id','is_new_session'])
     .map_batches(lambda x: pl.Series(np.cumsum(x.struct.field('is_new_session'))))
     .over('user_id')
     .alias('session_number'))
])

In [22]:
# Create a unique session string: user_id + session_number
events_cleaned = events_cleaned.with_columns(
    pl.concat_str([pl.col('user_id'),pl.lit('_'),pl.col('session_number')]).alias('session_id')
).drop(['is_new_session','session_number'])

In [23]:
# Final Check: Remove any row with missing critical IDs
events_cleaned = events_cleaned.filter(pl.col('user_id').is_not_null())

In [24]:
display(events_cleaned.head())

event_id,user_id,event_name,track_id,timestamp,device_id,device_type,session_id
str,str,str,str,datetime[μs],str,str,str
"""3f9a7558-f331-4d69-ad95-867445…","""0""","""Search""","""8acd631a""",2026-01-07 18:31:00,"""device_0""","""Mobile""","""0_1"""
"""7a3ed92c-2c12-48ea-9f97-50d2e8…","""0""","""Add to Playlist""","""22cd02e6""",2026-01-11 11:14:00,"""device_0""","""Mobile""","""0_2"""
"""98dd63ce-8df6-4983-a128-f85b6d…","""0""","""Play""","""32e67f38""",2026-01-20 12:55:00,"""device_0""","""Mobile""","""0_3"""
"""1db160c8-3bb9-42d3-98f6-071926…","""0""","""Play""","""ec268997""",2026-01-24 23:44:00,"""device_0""","""Mobile""","""0_4"""
"""b3af6fb6-5999-4ec4-95fc-472132…","""0""","""Search""","""d6139712""",2026-01-28 08:20:00,"""device_0""","""Desktop""","""0_5"""


In [27]:
print(f'Cleaned Events: {events_cleaned.height}')
print(f'Example Sessionization:\n{events_cleaned.select(['user_id','timestamp','session_id']).head(10)}')

Cleaned Events: 210159
Example Sessionization:
shape: (10, 3)
┌─────────┬─────────────────────┬────────────┐
│ user_id ┆ timestamp           ┆ session_id │
│ ---     ┆ ---                 ┆ ---        │
│ str     ┆ datetime[μs]        ┆ str        │
╞═════════╪═════════════════════╪════════════╡
│ 0       ┆ 2026-01-07 18:31:00 ┆ 0_1        │
│ 0       ┆ 2026-01-11 11:14:00 ┆ 0_2        │
│ 0       ┆ 2026-01-20 12:55:00 ┆ 0_3        │
│ 0       ┆ 2026-01-24 23:44:00 ┆ 0_4        │
│ 0       ┆ 2026-01-28 08:20:00 ┆ 0_5        │
│ 0       ┆ 2026-02-01 13:00:00 ┆ 0_6        │
│ 0       ┆ 2026-02-03 15:42:00 ┆ 0_7        │
│ 0       ┆ 2026-02-05 11:19:00 ┆ 0_8        │
│ 0       ┆ 2026-02-05 23:12:00 ┆ 0_9        │
│ 0       ┆ 2026-02-06 11:29:00 ┆ 0_10       │
└─────────┴─────────────────────┴────────────┘


In [29]:
# Exporting for BigQuery
users.write_csv('users_raw.csv')
tracks.write_csv('tracks_raw.csv')
events_cleaned.write_csv('events_raw.csv')

print('Files exported')

Files exported
